# Protocol effects on a second, architecturally different denoiser (CPU)

`denoise-protocol-effects` measured how scoring choices move a *fixed* denoiser, using DMID-d's released outputs. One
model is an illustration; two make it a budget. This run repeats the measurement with Restormer (CVX-based transformer,
σ-specific colour weights), on our own noise draws, under:
* full image vs GRL's crop region (top-left multiple of 8; Kodak central 500 cut to 496);
* RGB vs BT.601 Y channel;
* unclipped float input vs clipped-and-quantised uint8 input (the NTIRE convention).

Purpose: does the protocol effect transfer across architectures, or is it a property of one model?
Kaggle: CPU, internet on.

In [ ]:
import importlib.util, io, math, os, re, time, urllib.request, zipfile
import numpy as np
import pandas as pd
import torch
from PIL import Image

torch.set_num_threads(os.cpu_count())
SIGMAS = [15, 25, 50]
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
T0 = time.time()


def download(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    for attempt in range(5):
        try:
            with urllib.request.urlopen(url, timeout=300) as r, open(path, "wb") as f:
                while chunk := r.read(1 << 23):
                    f.write(chunk)
            return path
        except Exception as e:
            print(f"  retry {attempt + 1}: {e}", flush=True)
            time.sleep(5 * (attempt + 1))
    raise RuntimeError(url)


GH = "https://github.com/{}/releases/download/{}/{}"
arch = download("https://raw.githubusercontent.com/swz30/Restormer/68dc6ac472db26f16361150cb7a96a1bc87da93f/basicsr/models/archs/restormer_arch.py",
                f"{CACHE}/restormer_arch.py")
spec = importlib.util.spec_from_file_location("restormer_arch", arch)
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
Restormer = mod.Restormer

zs = download("https://github.com/Li-Tong-621/DMID/releases/download/v1.0/Datasets.zip", f"{CACHE}/Datasets.zip")


def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


CLEAN = {}
with zipfile.ZipFile(zs) as z:
    for ds in ("CBSD68", "Kodak24", "McMaster", "Urban100"):
        names = sorted([n for n in z.namelist() if n.startswith(f"Datasets/Gaussion/{ds}/") and not n.endswith("/")], key=natkey)
        CLEAN[ds] = [(n.split("/")[-1], np.asarray(Image.open(io.BytesIO(z.read(n))).convert("RGB"))) for n in names]
print({k: len(v) for k, v in CLEAN.items()}, f"({(time.time() - T0) / 60:.1f} min)")

In [ ]:
SETS = ["McMaster", "Kodak24", "CBSD68", "Urban100"]   # noise-seed order of diffusion-tweedie-benchmark


def noisy_of(ds, idx, sigma):
    """DMID's protocol: sample std exactly σ/255, unclipped."""
    seed = 1_000_000 * (SETS.index(ds) + 1) + 1000 * sigma + idx
    x = torch.from_numpy(CLEAN[ds][idx][1]).permute(2, 0, 1).float() / 255
    n = torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))
    return x + (sigma / 255) * n / n.std(unbiased=False)


def ycbcr_y(rgb):
    return 16.0 + (65.481 * rgb[..., 0] + 128.553 * rgb[..., 1] + 24.966 * rgb[..., 2]) / 255.0


def psnr(a, b):
    mse = np.mean((np.asarray(a, np.float64) - np.asarray(b, np.float64)) ** 2)
    return 10 * math.log10(255 ** 2 / mse) if mse > 0 else float("inf")


def grl_crop(ds, h, w):
    if ds == "Kodak24":
        i, j = (h - 500) // 2, (w - 500) // 2
        return slice(i, i + 496), slice(j, j + 496)
    return slice(0, h // 8 * 8), slice(0, w // 8 * 8)


@torch.no_grad()
def restormer(net, y):
    """Restormer's own test script: reflect-pad to a multiple of 8, run, crop back, clamp to [0, 1]."""
    h, w = y.shape[-2:]
    yp = torch.nn.functional.pad(y[None], (0, (-w) % 8, 0, (-h) % 8), mode="reflect")
    return net(yp)[0, :, :h, :w].clamp(0, 1)

In [ ]:
rows = []
for s in SIGMAS:
    w = download(GH.format("swz30/Restormer", "v1.0", f"gaussian_color_denoising_sigma{s}.pth"), f"{CACHE}/restormer_color_{s}.pth")
    net = Restormer(inp_channels=3, out_channels=3, dim=48, num_blocks=[4, 6, 6, 8], num_refinement_blocks=4,
                    heads=[1, 2, 4, 8], ffn_expansion_factor=2.66, bias=False, LayerNorm_type="BiasFree", dual_pixel_task=False)
    net.load_state_dict(torch.load(w, map_location="cpu", weights_only=True)["params"], strict=True)
    net.eval().requires_grad_(False)
    for ds in ("CBSD68", "Kodak24", "McMaster", "Urban100"):
        for idx, (name, clean) in enumerate(CLEAN[ds]):
            y = noisy_of(ds, idx, s)
            if ds == "Urban100" and s == 50:
                continue   # 100 images × ~1 MP on 4 CPU cores: skip the slowest cell
            variants = {"float": y,
                        "clipped uint8": torch.from_numpy(np.round(np.clip(y.permute(1, 2, 0).numpy(), 0, 1) * 255) / 255.0).permute(2, 0, 1).float()}
            for vin, x in variants.items():
                out = (restormer(net, x).permute(1, 2, 0).numpy() * 255).round()
                h, wd = clean.shape[:2]
                si, sj = grl_crop(ds, h, wd)
                c = clean
                rows.append(dict(dataset=ds, image=name, idx=idx, sigma=s, model="Restormer", input=vin, protocol="full", channel="RGB",
                                 psnr=psnr(out, c)))
                rows.append(dict(dataset=ds, image=name, idx=idx, sigma=s, model="Restormer", input=vin, protocol="GRL crop", channel="RGB",
                                 psnr=psnr(out[si, sj], c[si, sj])))
                if vin == "float":
                    rows.append(dict(dataset=ds, image=name, idx=idx, sigma=s, model="Restormer", input=vin, protocol="full", channel="Y",
                                     psnr=psnr(ycbcr_y(out), ycbcr_y(c))))
                    rows.append(dict(dataset=ds, image=name, idx=idx, sigma=s, model="Restormer", input=vin, protocol="GRL crop", channel="Y",
                                     psnr=psnr(ycbcr_y(out[si, sj]), ycbcr_y(c[si, sj]))))
        print(f"σ {s} {ds}: done ({(time.time() - T0) / 60:.1f} min)", flush=True)
        pd.DataFrame(rows).to_csv(f"{OUT}/restormer_protocols.csv", index=False)

R = pd.DataFrame(rows)
R.to_csv(f"{OUT}/restormer_protocols.csv", index=False)
M = R.pivot_table(index=["dataset", "sigma"], columns=["input", "channel", "protocol"], values="psnr", aggfunc="mean")
print("\nRestormer mean PSNR")
print(M.round(3).to_string())
base = M[("float", "RGB", "full")]
shift = M.sub(base, axis=0)
print("\nshift against the unclipped full-image RGB score (dB)")
print(shift.round(3).to_string())
key = shift[[("float", "RGB", "GRL crop"), ("float", "Y", "full"), ("clipped uint8", "RGB", "full")]]
key.columns = ["crop (RGB)", "Y channel", "clipped uint8 input"]
print("\nheadline shifts")
print(key.round(3).to_string())
with open(f"{OUT}/summary.txt", "w") as f:
    f.write(M.round(3).to_string() + "\n\n" + key.round(3).to_string())
print(f"done in {(time.time() - T0) / 60:.1f} min")